## 02. Text Preprocessing Pipeline

This notebook implements and validates the text normalization and regex cleaning pipeline for Cernovalis.

1. **Wire artifact stripping**: Remove leading `(Reuters)` wire service datelines from True articles to prevent trivial leakage.
2. **Media & tag removal**: Strip brackets/markers like `[VIDEO]`, `[WATCH]`, and HTML tags.
3. **URL & Email stripping**: Eliminate hyperlinked targets and contact addresses.
4. **Contraction expansion**: Standardize colloquialisms (`don't` -> `do not`, `it's` -> `it is`).
5. **ASCII / Unicode normalization**: Normalize accents and special glyphs.
6. **Case normalization & whitespace collapsing**: Produce consistent clean token streams.
7. **Dataset preparation**: Export cleaned corpus to `backend/ml/data/processed/cleaned_articles.csv`.

In [1]:
import sys
from pathlib import Path

# Resolve project root (Cernovalis) from notebook directory
curr = Path.cwd().resolve()
for p in [curr, curr.parent, curr.parent.parent, curr.parent.parent.parent]:
    if (p / 'backend' / 'app').is_dir():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

import pandas as pd
from backend.app.services.preprocessor import clean_text_regex, expand_contractions, remove_wire_artifacts

## 1. Load Raw Datasets

Load `True.csv` and `Fake.csv`, assign binary labels (`1` for Real/True, `0` for Fake), and concatenate into a working DataFrame.

In [2]:
candidates = (Path('../data/raw'), Path('backend/ml/data/raw'))
data_dir = next((p for p in candidates if p.is_dir()), None)
if data_dir is None:
    raise FileNotFoundError('Could not locate backend/ml/data/raw')

true_df = pd.read_csv(data_dir / 'True.csv')
fake_df = pd.read_csv(data_dir / 'Fake.csv')

true_df['label'] = 1
true_df['class'] = 'True'
fake_df['label'] = 0
fake_df['class'] = 'Fake'

df = pd.concat([true_df, fake_df], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f'Total combined articles: {len(df):,}')
df.head(3)

Total combined articles: 44,898


,title,text,subject,date,label,class
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",0,Fake
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",0,Fake
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",0,Fake


## 2. Inspect Cleaning Steps on Sample Texts

Verify that our pipeline properly handles wire datelines, HTML, URLs, and contractions.

In [3]:
sample_texts = [
    'WASHINGTON (Reuters) - The U.S. Senate voted 51-49 on Friday to pass tax overhaul legislation.',
    'BREAKING: [VIDEO] Hillary Clinton\'s campaign couldn\'t believe this link https://t.co/abc1234 or email info@campaign.org <p>Shocking update!</p>',
    'SEATTLE/WASHINGTON (Reuters) - Amazon said it\'s opening a new facility.',
]

for i, raw in enumerate(sample_texts, 1):
    cleaned = clean_text_regex(raw)
    print(f'Sample {i} RAW:\n  {raw}')
    print(f'Sample {i} CLEANED:\n  {cleaned}\n')

Sample 1 RAW:
  WASHINGTON (Reuters) - The U.S. Senate voted 51-49 on Friday to pass tax overhaul legislation.
Sample 1 CLEANED:
  the u s senate voted on friday to pass tax overhaul legislation

Sample 2 RAW:
  BREAKING: [VIDEO] Hillary Clinton's campaign couldn't believe this link https://t.co/abc1234 or email info@campaign.org <p>Shocking update!</p>
Sample 2 CLEANED:
  breaking hillary clinton s campaign could not believe this link or email info shocking update

Sample 3 RAW:
  SEATTLE/WASHINGTON (Reuters) - Amazon said it's opening a new facility.
Sample 3 CLEANED:
  amazon said it is opening a new facility



## 3. Apply Cleaning Pipeline to Dataset

We generate clean versions of both the headline (`title_clean`) and the body (`text_clean`), as well as a combined representation (`content_clean`).

In [4]:
# Clean titles
df['title_clean'] = df['title'].fillna('').apply(lambda t: clean_text_regex(t, strip_wire=False))

# Clean article text
df['text_clean'] = df['text'].fillna('').apply(clean_text_regex)

# Combined full representation
df['content_clean'] = (df['title_clean'] + ' ' + df['text_clean']).str.strip()

print('Sample cleaned records:')
df[['title', 'title_clean', 'text_clean']].head(3)

Sample cleaned records:


,title,title_clean,text_clean
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,breaking gop chairman grassley has had enough ...,donald trump s white house is in chaos and the...
1,Failed GOP Candidates Remembered In Hilarious...,failed gop candidates remembered in hilarious ...,now that donald trump is the presumptive gop n...
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,mike pence s new dc neighbors are hilariously ...,mike pence is a huge homophobe he supports ex ...


## 4. Verification and Sanity Checks

Check length distributions, verify wire service removal, and quantify empty articles.

In [5]:
print('Reuters wire tag check in text_clean:')
reuters_count = df['text_clean'].str.contains(r'\(reuters\)|\breuters\b', regex=True).sum()
print(f'Articles with reuters in text_clean: {reuters_count:,} ({reuters_count / len(df):.2%})')

print('\nEmpty cleaned texts:')
empty_titles = (df['title_clean'] == '').sum()
empty_bodies = (df['text_clean'] == '').sum()
empty_content = (df['content_clean'] == '').sum()
print(f'Empty title_clean: {empty_titles}')
print(f'Empty text_clean: {empty_bodies}')
print(f'Empty content_clean: {empty_content}')

Reuters wire tag check in text_clean:


Articles with reuters in text_clean: 5,314 (11.84%)

Empty cleaned texts:
Empty title_clean: 9
Empty text_clean: 715
Empty content_clean: 9


## 5. Export Cleaned Dataset

Save the preprocessed corpus to `backend/ml/data/processed/cleaned_articles.csv` for downstream feature engineering and modeling.

In [6]:
output_candidates = (Path('../data/processed'), Path('backend/ml/data/processed'))
out_dir = next((p for p in output_candidates if p.is_dir()), None)
if out_dir is None:
    out_dir = Path('backend/ml/data/processed')
    out_dir.mkdir(parents=True, exist_ok=True)

export_cols = ['title', 'text', 'subject', 'date', 'label', 'class', 'title_clean', 'text_clean', 'content_clean']
out_path = out_dir / 'cleaned_articles.csv'
df[export_cols].to_csv(out_path, index=False, encoding='utf-8')
print(f'Successfully saved cleaned dataset to {out_path.resolve()} ({len(df):,} rows)')

Successfully saved cleaned dataset to C:\Users\HP\Documents\Cernovalis\backend\ml\data\processed\cleaned_articles.csv (44,898 rows)
